In [1]:
import os
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ['HF_HOME'] = '/cache_model/huggingface/' # Trỏ đến thư mục cache chứa model của bạn

In [4]:
# Block này sẽ train một mô hình cho toàn bộ domain. 
# Xem thêm trong file train_t5_acste.py
# File submission sẽ được lưu trong `outputs/t5-small/predictions_PublicTest.json`
# Bạn chỉ cần submit file này là được. Không cần xử lý gì thêm.
# Lưu ý set đúng file model trên local và --pretrained_local_dir
!python train_t5_acste.py \
    --pretrained_local_dir "/cache_model/huggingface/hub/models--google-t5--t5-small/snapshots/df1b051c49625cf57a3d0d8d3863ed4d13564fe4/" \
    --model_name google-t5/t5-small \
    --dataset /home/user/TACVU1 \
    --dataset_layout merged \
    --predict_split PublicTest \
    --train_all_domains \
    --domain all \
    --output_dir "outputs/t5-small" \
    --num_train_epochs 3 \
    --seed 42


Loading weights: 100%|█████████████████████| 131/131 [00:00<00:00, 11529.34it/s]
Đã ghi 1802 câu (5 domain) tới outputs/t5-small/predictions_PrivateTest.json


In [ ]:
# Đối với tập PrivateTest, bạn có thể cần chạy lại script này với predict_split = PrivateTest với --predict_only flag và check point model tốt nhất
# !python train_t5_acste.py \
#     --pretrained_local_dir "/cache_model/huggingface/hub/models--google-t5--t5-small/snapshots/df1b051c49625cf57a3d0d8d3863ed4d13564fe4/" \
#     --predict_only --checkpoint "outputs/t5-small/checkpoint-best" \
#     --model_name google-t5/t5-small \
#     --dataset /home/user/TACVU1 \
#     --dataset_layout merged \
#     --predict_split PrivateTest \
#     --train_all_domains \
#     --domain all \
#     --output_dir "outputs/t5-small" \
#     --num_train_epochs 3 \
#     --seed 42

In [5]:
# Sau đó Đổi tên file submission cho đúng định dạng submission.json
# Và zip file submission và code lại.

!mv outputs/t5-small/predictions_PublicTest.json submission.json
!zip -r submission.zip submission.json train_t5_acste.py main.ipynb

updating: train_t5_acste.py (deflated 73%)
updating: main.ipynb (deflated 72%)
updating: submission.json (deflated 87%)


In [6]:
# Block này sẽ train một mô hình cho từng domain. Xem thêm trong file train_t5_acste.py
# Thay đổi `--domain` thành tên domain tương ứng. e.g. `Restaurant`, `Laptop`, `Hotel`, `Books`, `Clothing`
# và `--output_dir` thành thư mục lưu kết quả cho domain đó.
# Mỗi domain sẽ tạo ra file prediction tương ứng
# Đối với tập PrivateTest, bạn có thể cần chạy lại script này với predict_split = PrivateTest với --predict_only flag và check point model tốt nhất
!python train_t5_acste.py \
    --pretrained_local_dir "/cache_model/huggingface/hub/models--google-t5--t5-small/snapshots/df1b051c49625cf57a3d0d8d3863ed4d13564fe4/" \
    --model_name google-t5/t5-base \
    --dataset /home/user/TACVU1 \
    --dataset_layout merged \
    --predict_split PublicTest \
    --domain Restaurant \
    --output_dir "outputs/t5-small-single/Restaurant" \
    --num_train_epochs 5 \
    --seed 42


Map: 100%|██████████████████████████| 510/510 [00:00<00:00, 30331.45 examples/s]
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
{'loss': '5.019', 'grad_norm': '15.73', 'learning_rate': '0.0001081', 'epoch': '0.1104'}
{'loss': '1.544', 'grad_norm': '2.568', 'learning_rate': '0.0002184', 'epoch': '0.2208'}
{'loss': '0.5989', 'grad_norm': '2.657', 'learning_rate': '0.0002982', 'epoch': '0.3311'}
{'loss': '0.488', 'grad_norm': '3.14', 'learning_rate': '0.0002911', 'epoch': '0.4415'}
{'loss': '0.4669', 'grad_norm': '2.204', 'learning_rate': '0.0002841', 'epoch': '0.5519'}
{'loss': '0.4515', 'grad_norm': '1.897', 'learning_rate': '0.000277', 'epoch': '0.6623'}
{'loss': '0.4164', 'grad_norm': '1.44', 'learning_rate': '0.00027', 'epoch': '0.7726'}
{'loss': '0.4093', 'grad_norm': '1.53', 'learning_rate': '0.0002629', 'epoch': '0.883'}
{'loss': '0.4049', 'grad_norm': '1.371', 'learning_rate': '0.0002559', 'epoch': '0.9934'}
 73%|██████████████████████████████

In [ ]:
# Sau khi chạy xong 5 domain ta merge các file prediction từ các domain.
# File submission sau khi merge sẽ được lưu trong `outputs/t5-small/submission.json`
test_split='PublicTest'
os.makedirs(f"outputs/t5-small-single/{test_split}/", exist_ok=True)

for domain in ['Books', 'Clothing', 'Hotel', 'Laptop', 'Restaurant']:
    os.system(f"cp outputs/t5-small-single/{domain}/{test_split}/{domain}.json outputs/t5-small-single/{test_split}")


In [ ]:
# Chạy merge file
!python merge_domain_submission.py --input_dir outputs/t5-small-single/PublicTest --output outputs/t5-small-single/submission.json

# Cuối cùng zip file submission và code lại
# !zip -r submission.zip outputs/t5-small-single/submission.json train_t5_acste.py merge_domain_submission.py main.ipynb

In [7]:
# Script này sẽ chạy evaluation trên tập Dev. 
# Cũng là script để chạy đánh giá cho bộ dữ liệu Public và Private Test.
!python evaluation_script.py \
  --submission example.json \
  --domain all \
  --gold_merged /home/data/TACVU1/train/Dev.json

ACSTE Evaluation (Micro-F1, exact triple match)

Restaurant:
  Precision: 0.5858
  Recall:    0.5247
  Micro-F1:  0.5536
  (Gold: 709, Pred: 635, Correct: 372)

Laptop:
  Precision: 0.4136
  Recall:    0.3926
  Micro-F1:  0.4028
  (Gold: 512, Pred: 486, Correct: 201)

Hotel:
  Precision: 0.6461
  Recall:    0.5827
  Micro-F1:  0.6127
  (Gold: 520, Pred: 469, Correct: 303)

Books:
  Precision: 0.4625
  Recall:    0.3899
  Micro-F1:  0.4231
  (Gold: 395, Pred: 333, Correct: 154)

Clothing:
  Precision: 0.4453
  Recall:    0.3782
  Micro-F1:  0.4090
  (Gold: 312, Pred: 265, Correct: 118)

Final score (average Micro-F1 over 5 domains):
  Score:     0.4802
